In [1]:
import pandas as pd


def read_csv_with_fallback(path, encodings=("utf-8-sig", "utf-8", "gb18030", "gbk")):
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


train_path = "dataset/triage/label_1_cls_data/label_1_train.csv"
test_path = "dataset/triage/label_1_cls_data/label_1_test.csv"

train_data = read_csv_with_fallback(train_path)
test_data = read_csv_with_fallback(test_path)

print(train_data.head())
print(test_data.head())

                 titles                                              infos  \
0  早上刚醒左边门牙痛的厉害一点右边有一点痛                     早上刚醒左边门牙痛的厉害一点右边有一点痛我想问一下是什么原因   
1           精神崩溃时人有哪些表现                  精神崩溃时人有哪些表现精神崩溃时人有哪些表现精神崩溃时人有哪些表现   
2     来完月经肚子疼腰疼，2月份药流清宫  来完月经肚子疼腰疼，2月份药流清宫，复查过流的干净，但是这个月月经走后就肚子不舒服，前俩年得...   
3       喉咙抖了几下。就是抽筋的感觉。  刚才窝着坐在沙发上的时候喉咙突然抖了一两秒钟，不痛不痒的，就是有点吓到了，而且我一直喉咙有异...   
4      白带发黄，腥臭味儿，瘙痒。。。。            月经前后会有腥臭味儿，现在月经快完了腥臭味儿又有了。是什么炎症吗，白带发黄瘙痒   

  labels_1_word  labels_1  
0           五官科        11  
1         心理健康科         3  
2           妇产科         5  
3           五官科        11  
4           妇产科         5  
                 titles                                              infos  \
0  用什么药可去除泪管瘤，激光据说会留疤。有                  不疼不痒，影响美观，。是不是治疗皮肤癌的药可以退去呢？有点不敢用。   
1          泌乳素高.去挂什么科rt          wt泌乳素高，不知道去医院挂什么科，拍了脑部磁共振，乳房有粘的分泌物.GYKZBZ   
2  宝宝半个月了，因为上火睡觉总是惊醒必须抱  宝宝半个月了，因为上火睡觉总是惊醒必须抱着才能继续入睡，但是一直抱着对他的身体发宵会有影响吗...   
3      宝宝脖子歪向右侧，脖子处无硬块。                4个月大的婴儿，竖抱时头

In [2]:
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "freedomking/mc-bert"
WORKSPACE_DIR = Path(".").resolve()
LOCAL_MODEL_DIR = WORKSPACE_DIR / "models" / "mc-bert"
LOCAL_CACHE_DIR = WORKSPACE_DIR / ".hf-cache"

LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def has_local_model(model_dir: Path) -> bool:
    has_config = (model_dir / "config.json").exists()
    has_vocab = (model_dir / "vocab.txt").exists()
    has_weights = (model_dir / "pytorch_model.bin").exists() or (model_dir / "model.safetensors").exists()
    return has_config and has_vocab and has_weights


if not has_local_model(LOCAL_MODEL_DIR):
    try:
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=str(LOCAL_MODEL_DIR),
            cache_dir=str(LOCAL_CACHE_DIR),
            local_dir_use_symlinks=False,
            resume_download=True,
        )
    except Exception as exc:
        raise RuntimeError(
            f"首次下载失败，请检查网络/代理，或手动把模型文件放到: {LOCAL_MODEL_DIR}"
        ) from exc

# Enforce loading from workspace-local model directory only.
tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL_DIR), local_files_only=True)
model = AutoModel.from_pretrained(str(LOCAL_MODEL_DIR), torch_dtype="auto", local_files_only=True)

print(f"Model dir: {LOCAL_MODEL_DIR}")
print(f"Cache dir (workspace-local): {LOCAL_CACHE_DIR}")

/root/miniconda3/envs/transfomers/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model dir: /home/projects/AI-Inquiry/models/mc-bert
Cache dir (workspace-local): /home/projects/AI-Inquiry/.hf-cache


In [3]:
# 计算 train_data["infos"] 的 token 长度分布（按 tokenizer）
texts = train_data["infos"].fillna("").astype(str)

def get_token_lengths(series, batch_size=2048):
    lengths = []
    for i in range(0, len(series), batch_size):
        batch = series.iloc[i:i + batch_size].tolist()
        encoded = tokenizer(batch, add_special_tokens=True, truncation=False)
        lengths.extend(len(ids) for ids in encoded["input_ids"])
    return pd.Series(lengths, index=series.index, name="token_len")

token_len = get_token_lengths(texts)

# 基本统计
print(token_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("max token length:", token_len.max())

# 分桶分布
bins = [0, 16, 32, 64, 128, 256, 512, 1024, float("inf")]
labels = ["0-16", "17-32", "33-64", "65-128", "129-256", "257-512", "513-1024", "1025+"]

dist = pd.cut(token_len, bins=bins, labels=labels, include_lowest=True).value_counts(sort=False)
dist_df = pd.DataFrame({
    "count": dist,
    "ratio": (dist / len(token_len)).round(4)
})

print(dist_df)

count    373686.000000
mean         59.028899
std          43.523397
min           3.000000
50%          47.000000
90%         101.000000
95%         130.000000
99%         224.000000
max        2620.000000
Name: token_len, dtype: float64
max token length: 2620
            count   ratio
token_len                
0-16         4955  0.0133
17-32       57353  0.1535
33-64      206566  0.5528
65-128      85321  0.2283
129-256     17030  0.0456
257-512      2233  0.0060
513-1024      210  0.0006
1025+          18  0.0000


In [4]:
def calc_class_ratio(df):
    result = (
        df.groupby(["labels_1", "labels_1_word"], dropna=False)
          .size()
          .reset_index(name="count")
          .sort_values("labels_1")
          .reset_index(drop=True)
    )
    result["ratio"] = (result["count"] / result["count"].sum()).round(4)
    result["percent"] = (result["ratio"] * 100).round(2).astype(str) + "%"
    return result

train_class_ratio = calc_class_ratio(train_data)
test_class_ratio = calc_class_ratio(test_data)

print("train_data 各类别占比：")
display(train_class_ratio)

print("test_data 各类别占比：")
display(test_class_ratio)

train_data 各类别占比：


,labels_1,labels_1_word,count,ratio,percent
0,0,肿瘤科,6211,0.0166,1.66%
1,1,整形美容科,11250,0.0301,3.01%
2,2,康复医学科,554,0.0015,0.15%
3,3,心理健康科,9752,0.0261,2.61%
4,4,儿科,19517,0.0522,5.22%
5,5,妇产科,83803,0.2243,22.43%
6,6,传染科,8420,0.0225,2.25%
7,7,皮肤性病科,44080,0.1180,11.8%
8,8,辅助检查科,58,0.0002,0.02%
9,9,内科,84261,0.2255,22.55%


test_data 各类别占比：


,labels_1,labels_1_word,count,ratio,percent
0,0,肿瘤科,1080,0.0164,1.64%
1,1,整形美容科,2018,0.0306,3.06%
2,2,康复医学科,118,0.0018,0.18%
3,3,心理健康科,1650,0.0250,2.5%
4,4,儿科,3430,0.0520,5.2%
5,5,妇产科,14757,0.2238,22.38%
6,6,传染科,1529,0.0232,2.32%
7,7,皮肤性病科,7720,0.1171,11.71%
8,8,辅助检查科,9,0.0001,0.01%
9,9,内科,14661,0.2223,22.23%
